# 03 趋势与漏斗

## 本步目标

1. 看最近 12 个月的进件量和审批通过率趋势。
2. 拆 2026-09 的审批漏斗。
3. 对比渠道和产品，找出“规模大但通过率低”的渠道。

## 看图顺序

先看总量趋势，再看漏斗，最后看渠道/产品。不要一上来就下“风控变严”的结论。

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from IPython.display import display
from src.db import query_df
from src.charts import monthly_trend, channel_bar, contribution_bar, risk_scatter

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_columns", 50)
FIG = ROOT / "outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

## 1. 最近 12 个月趋势

In [2]:
trend = query_df('''
SELECT * FROM `v_申请审批月度`
WHERE `月份` >= '2025-10'
ORDER BY `月份`
''')
display(trend)
monthly_trend(trend, FIG / "01_月度趋势.png")

,月份,进件量,审批通过量,实际用信量,拒绝量,取消量,未使用额度量,审批通过率,用信率,审批金额,件均
0,2025-10,54766,24579.0,23980.0,11067.0,19120.0,599.0,0.4488,0.9756,571457.21,23.83
1,2025-11,54749,22609.0,22071.0,11138.0,21002.0,538.0,0.4130,0.9762,523151.27,23.70
2,2025-12,60101,23283.0,22709.0,12164.0,24654.0,574.0,0.3874,0.9753,538882.08,23.73
3,2026-01,62997,22760.0,22168.0,13249.0,26988.0,592.0,0.3613,0.9740,537746.04,24.26
4,2026-02,56183,18918.0,18393.0,12458.0,24807.0,525.0,0.3367,0.9722,445853.38,24.24
5,2026-03,61814,19290.0,18680.0,14194.0,28330.0,610.0,0.3121,0.9684,463670.04,24.82
6,2026-04,52772,16487.0,15880.0,12140.0,24145.0,607.0,0.3124,0.9632,358618.89,22.58
7,2026-05,44347,13846.0,13223.0,9266.0,21235.0,623.0,0.3122,0.9550,288200.83,21.80
8,2026-06,34640,11359.0,10738.0,5609.0,17672.0,621.0,0.3279,0.9453,230742.85,21.49
9,2026-07,24967,9148.0,8598.0,1902.0,13917.0,550.0,0.3664,0.9399,161115.52,18.74


WindowsPath('E:/ai数据分析工作/consumer-finance-approval-analysis/outputs/figures/01_月度趋势.png')

## 2. 2026-09 漏斗

In [3]:
funnel = query_df("SELECT * FROM `v_申请审批月度` WHERE `月份` = '2026-09'")
display(funnel.T)

,0
月份,2026-09
进件量,36012
审批通过量,8245.0
实际用信量,8070.0
拒绝量,6827.0
取消量,20940.0
未使用额度量,175.0
审批通过率,0.229
用信率,0.9788
审批金额,188680.95


## 3. 2026-09 渠道对比

In [4]:
channel = query_df('''
SELECT `渠道类型`, `进件量`, `审批通过量`, `审批通过率`, `用信率`, `审批金额`, `件均`
FROM `v_渠道月度`
WHERE `月份` = '2026-09'
ORDER BY `进件量` DESC
''')
display(channel)
channel_bar(channel, FIG / "02_渠道进件与通过率.png")

,渠道类型,进件量,审批通过量,审批通过率,用信率,审批金额,件均
0,Credit and cash offices,23430,1837.0,0.0784,1.0000,110701.40,60.26
1,Contact center,4939,214.0,0.0433,1.0000,14296.85,66.81
2,Country-wide,4384,3842.0,0.8764,0.9545,33625.32,9.17
3,Stone,1397,1231.0,0.8812,1.0000,14169.40,11.51
4,Regional / Local,1075,940.0,0.8744,1.0000,10505.76,11.18
5,AP+ (Cash loan),468,108.0,0.2308,1.0000,3859.70,35.74
6,Channel of corporate sales,319,73.0,0.2288,1.0000,1522.54,20.86


WindowsPath('E:/ai数据分析工作/consumer-finance-approval-analysis/outputs/figures/02_渠道进件与通过率.png')

## 4. 产品类型对比

In [5]:
product = query_df('''
SELECT COALESCE(product_type,'未知产品') AS 产品类型,
       COUNT(*) AS 进件量,
       ROUND(SUM(status IN ('Approved','Unused offer')) / COUNT(*), 4) AS 审批通过率,
       ROUND(SUM(CASE WHEN status='Approved' THEN approved_amount ELSE 0 END), 2) AS 审批金额
FROM `v_申请明细`
WHERE DATE_FORMAT(decision_date, '%Y-%m') = '2026-09'
GROUP BY 1
ORDER BY 进件量 DESC
''')
display(product)

,产品类型,进件量,审批通过率,审批金额
0,XNA,26727,0.2090,52101.37
1,x-sell,7891,0.2905,125957.06
2,walk-in,1394,0.2633,10622.53


## 本步结论

2026-09 进件量 36,012 笔（环比 +26.2%），审批通过率 22.90%（环比 -6.5pp）；用信率 97.88%，说明“批而不用”不是主要矛盾。